# 06 · Type Hints & Static Type Checking 🆕

**Module 05 – Functions** · Estimated time: 60 min

> ❗ **Missing from both the bootcamp and the handbook.** Type hints are now standard in professional Python code: every major library, IDE autocompletion, FastAPI, Pydantic and dataclasses use them.

### 🎯 What you will learn
- What type hints are (and what they are **not**)
- Annotating variables, parameters and return values
- Built-in generics `list[int]`, `dict[str, float]` (3.9+)
- Union `X | Y` and optional values `X | None` (3.10+)
- `Callable`, `Iterable`, `Sequence`, `Mapping`, `Any`, `Literal`, `Final`, `TypedDict`
- 🆕 The `type` statement and generic syntax `def f[T](...)` (3.12)
- 🆕 `Self` (3.11), deferred annotations (3.14)
- Running a type checker (`mypy` / `pyright`)
- ⚠️ Old-style `typing.List`, `Optional`, `Union`

## 1. What are type hints?

Annotations that describe the **expected types**. They are:
- ✅ Documentation for humans
- ✅ Information for IDEs (autocomplete, errors as you type)
- ✅ Checkable by tools such as **mypy**, **pyright** and Pylance
- ❌ **Not enforced at runtime** by Python itself

In [1]:
def greet(name: str, times: int = 1) -> str:
    return ", ".join([f"Hello {name}"] * times)

print(greet("Asha", 2))
print(greet(42))          # Python doesn't stop this! Only a type checker would complain.

Hello Asha, Hello Asha
Hello 42


## 2. Basic annotations

In [2]:
age: int = 30
price: float = 9.99
name: str = "Ravi"
active: bool = True
nothing: None = None

def area(width: float, height: float) -> float:
    return width * height

def log(message: str) -> None:       # returns nothing
    print(message)

print(area.__annotations__)

{'width': <class 'float'>, 'height': <class 'float'>, 'return': <class 'float'>}


> 💡 `int` is accepted wherever `float` is expected (a typing convenience).

## 3. Collections (🆕 3.9+: use the built-in types directly)

In [3]:
scores: list[int] = [90, 85]
point: tuple[float, float] = (1.5, 2.0)
row: tuple[int, ...] = (1, 2, 3, 4)        # any length, all ints
tags: set[str] = {"python", "ai"}
prices: dict[str, float] = {"apple": 0.5}
matrix: list[list[int]] = [[1, 2], [3, 4]]

def average(nums: list[float]) -> float:
    return sum(nums) / len(nums)

print(average([1, 2, 3]))

2.0


> ⚠️ **Old (before 3.9):** `from typing import List, Dict, Tuple, Set` → `List[int]`, `Dict[str, int]`. These still work but are **deprecated aliases**. Use `list[int]`, `dict[str, int]`.

## 4. Unions and optional values (🆕 3.10+: the `|` operator)

In [4]:
def parse_id(value: int | str) -> int:
    return int(value)

def find_user(user_id: int) -> str | None:     # may return None
    users = {1: "Asha"}
    return users.get(user_id)

print(parse_id("7"), find_user(1), find_user(2))

7 Asha None


> ⚠️ **Old:** `Union[int, str]` and `Optional[str]` from `typing`. Prefer `int | str` and `str | None`.

## 5. Abstract collection types: accept broadly, return precisely

In [5]:
from collections.abc import Iterable, Sequence, Mapping, Callable, Iterator

def total(values: Iterable[float]) -> float:      # list, tuple, set, generator... all OK
    return sum(values)

def first(items: Sequence[str]) -> str:          # anything indexable
    return items[0]

def apply(func: Callable[[int], int], value: int) -> int:   # a function int → int
    return func(value)

def count_up(n: int) -> Iterator[int]:            # a generator
    yield from range(n)

print(total({1.5, 2.5}), first(("a", "b")), apply(lambda x: x * 2, 21), list(count_up(3)))

4.0 a 42 [0, 1, 2]


## 6. More useful types

In [6]:
from typing import Any, Literal, Final, TypedDict, NotRequired

def debug(value: Any) -> None:                    # opt out of checking
    print(repr(value))

def set_mode(mode: Literal["r", "w", "a"]) -> str:   # only these exact values
    return f"mode={mode}"

MAX_RETRIES: Final = 3                             # must not be reassigned

class Movie(TypedDict):                            # shape of a dict (e.g. JSON)
    title: str
    year: int
    rating: NotRequired[float]                     # optional key

m: Movie = {"title": "Inception", "year": 2010}
print(set_mode("r"), m)

mode=r {'title': 'Inception', 'year': 2010}


## 7. 🆕 Type aliases with the `type` statement (Python 3.12+)

In [7]:
type Vector = list[float]
type Matrix = list[Vector]
type JSON = dict[str, "JSON"] | list["JSON"] | str | int | float | bool | None   # recursive!

def scale(v: Vector, k: float) -> Vector:
    return [k * x for x in v]

print(scale([1.0, 2.0], 3))
print(Vector, Vector.__value__)

[3.0, 6.0]
Vector list[float]


> ⚠️ Older style: `Vector = list[float]` or `Vector: TypeAlias = list[float]`.

## 8. 🆕 Generics with the new syntax (Python 3.12+, PEP 695)

Write a function that works for *any* type while keeping the type relationship:

In [8]:
def first_item[T](items: list[T]) -> T:
    return items[0]

def pair_up[K, V](keys: list[K], values: list[V]) -> dict[K, V]:
    return dict(zip(keys, values))

class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

s = Stack[int]()
s.push(1); s.push(2)
print(first_item(["a", "b"]), pair_up(["x", "y"], [1, 2]), s.pop())

a {'x': 1, 'y': 2} 2


In [9]:
# Bounded / constrained type parameters
def biggest[T: (int, float)](a: T, b: T) -> T:    # T must be int or float
    return a if a > b else b
print(biggest(3, 7))

7


> ⚠️ **Old (pre-3.12):** `from typing import TypeVar, Generic` → `T = TypeVar("T")`, `class Stack(Generic[T]):`. You'll still see this a lot in existing code.

## 9. 🆕 `Self` (3.11) and forward references

In [10]:
from typing import Self

class Node:
    def __init__(self, value: int, next: Node | None = None) -> None:   # 'Node' used before it's fully defined
        self.value = value
        self.next = next

    def set_next(self, node: Node) -> Self:
        self.next = node
        return self

n = Node(1).set_next(Node(2))
print(n.value, n.next.value)

1 2


🆕 **Python 3.14 (PEP 649/749): annotations are evaluated lazily.** That's why `Node` above works *without* quotes, even though the class isn't finished yet. Before 3.14 you needed `"Node"` in quotes or `from __future__ import annotations`. That `__future__` import is no longer needed and is expected to be deprecated in a future version.

You can inspect annotations safely with the new `annotationlib` module:

In [11]:
import annotationlib

def f(x: UndefinedType) -> int:      # UndefinedType doesn't exist and that's fine until inspected
    return 1

print(annotationlib.get_annotations(f, format=annotationlib.Format.STRING))
print(annotationlib.get_annotations(f, format=annotationlib.Format.FORWARDREF))

{'x': 'UndefinedType', 'return': 'int'}
{'x': ForwardRef('UndefinedType', owner=<function f at 0x107a96350>), 'return': <class 'int'>}


## 10. Running a type checker

Type hints are checked by external tools. The two most popular:
- **mypy**: `pip install mypy` → `mypy your_file.py`
- **pyright** (also inside VS Code's *Pylance*): `pip install pyright` → `pyright your_file.py`

Let's write a file with a bug and let mypy find it **without running the code**:

In [12]:
code = '''
def total_price(prices: list[float], tax: float) -> float:
    return sum(prices) * (1 + tax)

total_price([10.0, 20.0], "0.18")     # bug: tax should be a float, not a str

def get_name(user: dict[str, str]) -> str:
    return user.get("name")             # bug: .get() may return None
'''
with open("typed_demo.py", "w", encoding="utf-8") as f:
    f.write(code)

import subprocess, sys
result = subprocess.run([sys.executable, "-m", "mypy", "typed_demo.py", "--no-error-summary", "--no-color-output"],
                        capture_output=True, text=True)
print(result.stdout)

typed_demo.py:5: error: Argument 2 to "total_price" has incompatible type "str"; expected "float"  [arg-type]
typed_demo.py:8: error: Incompatible return value type (got "str | None", expected "str")  [return-value]



## 11. Runtime checks when you *need* them

Hints don't validate data at runtime. For untrusted data (user input, JSON), validate explicitly or use a library like **Pydantic**:

In [13]:
def set_age(age: int) -> int:
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    return age

print(set_age(30))

30


> ➡️ **Next level:** generics in depth, `ParamSpec`, `@overload`, `TypedDict` extras, `TypeIs`, `assert_never`, `Annotated` and more are in **Module 10 · 07 Advanced Type Hints**. Runtime validation from type hints is **Pydantic** (Module 15).

## 📝 Summary

| Need | Modern syntax | ⚠️ Old syntax |
|---|---|---|
| list of ints | `list[int]` | `List[int]` |
| one of two types | `int \| str` | `Union[int, str]` |
| may be None | `str \| None` | `Optional[str]` |
| alias | `type Vec = list[float]` | `Vec = List[float]` |
| generic function | `def f[T](x: T) -> T` | `T = TypeVar("T")` |
| forward ref | `-> Node` (3.14) | `-> "Node"` |

- Hints are optional, not enforced at runtime, but hugely helpful with mypy/pyright and IDEs.

## 🏋️ Practice
Add type hints to these functions, then check them with mypy:
```python
def word_count(text):
    return len(text.split())

def merge(a, b):
    return {**a, **b}           # dicts of str → int

def find(items, target):
    for i, x in enumerate(items):
        if x == target:
            return i
    return None
```

<details><summary>Show solutions</summary>

```python
def word_count(text: str) -> int:
    return len(text.split())

def merge(a: dict[str, int], b: dict[str, int]) -> dict[str, int]:
    return {**a, **b}

def find[T](items: list[T], target: T) -> int | None:
    for i, x in enumerate(items):
        if x == target:
            return i
    return None
```
</details>